# Deriving the `BackgroundFit true_generator_q2` binning and prior widths from data

Everything in `parameterSet_backgroundFit_true_q2.yaml` and `GenerateBackgroundFitQ2Cov.C` right now (9 bins at 0.1 GeV$^2$ steps + one overflow bin, flat 40% Gaussian prior on every bin) was **borrowed from Howard et al.'s technote**, not derived from our own MC. This notebook redoes both choices from first principles, following the step-by-step procedure worked out in chat:

1. Freeze the population (categories 4,5,6,7 -- already fixed, not redone here).
2. Check whether the *bin edges* are supported by raw MC statistics, at both the current resolution and a finer one.
3. Check whether `true_W` is actually usable for a 2D (W, Q$^2$) binning the way Howard's is.
4. Check whether the true-level Q$^2$ binning has genuine separating power in the fit's actual reco observable (`reco_dpT_lp`), not just in raw truth-level statistics.
5. Derive a bin-by-bin prior width from an independent generator comparison (GENIE vs. NuWro), instead of a flat borrowed 40%.

**Top-line results (already run once against the real files -- see chat for the full numbers):** raw MC statistics are not the limiting factor even at 2x finer binning (worst-case relative MC stat. uncertainty 3.24% vs. a 10% floor); `true_W` is `NaN` for ~61-67% of background events, which rules out replicating Howard's 2D binning as-is; `reco_dpT_lp` does show a genuine monotonic shift across true Q$^2$ bins (not just overlapping noise), so the binning has real reco-level handle; and the GENIE-vs-NuWro comparison gives a strongly *non-flat* bin-by-bin disagreement (1.6% in the middle bins, up to 42% in the lowest-Q$^2$ bin -- exactly the bin our fake-data closure test pulled hardest).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob
import time
import uproot

plt.style.use("../style.mplstyle")

ICARUS_ROOT = '/Users/rvizarreta/Library/CloudStorage/GoogleDrive-rvizarreta14@gmail.com/My Drive/🏛 PhD Repository/🚀 Research/🤖 Experiments&Projects/ICARUS'

# Q2 bin edges currently used in binning_true_generator_q2.txt
CURRENT_EDGES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 99999.0]
BACKGROUND_CATEGORIES = [4, 5, 6, 7]  # same categories as parameterSet_backgroundFit_true_q2.yaml's applyCondition

def open_with_retry(path, tries=5, sleep_s=2.0):
    """The Google-Drive-mounted files occasionally hit a transient FUSE-level
    'Resource deadlock avoided' OSError on first read -- retrying a few times
    with a short sleep clears it. Not related to uproot itself."""
    last_exc = None
    for _ in range(tries):
        try:
            return uproot.open(path)
        except OSError as exc:
            last_exc = exc
            time.sleep(sleep_s)
    raise last_exc

## Step 1: population

Already fixed elsewhere (`applyCondition: "([category]==4) || ([category]==5) || ([category]==6) || ([category]==7)"`, matching the Q2/Pion fake-data reweights) -- not redone here.

## Step 2/3: are the bin edges actually supported by MC statistics?

Load `category`, `true_generator_q2`, `true_W`, `ppfx_cv_weight` from **both** the `selected` and `sideband` trees (the `BackgroundFit` parameter set applies to `"*"` datasets, so both feed its constraint), combine them, and histogram the background population's Q$^2$ in whatever candidate bin edges we're testing. Statistical uncertainty per bin uses the standard weighted-histogram estimator $\sigma_i = \sqrt{\sum w_j^2}$ (not $\sqrt{N}$), since `ppfx_cv_weight` varies event to event.

In [ ]:
MC_PATH = f'{ICARUS_ROOT}/ICARUS_CC0pi_Selection/data/icarus_numi_numu_mc_onbeam_offbeam_syst_gundam.root'
f_mc = open_with_retry(MC_PATH)

branches = ["category", "true_generator_q2", "true_W", "reco_dpT_lp", "cut_type", "reco_leading_muon_containment", "ppfx_cv_weight"]

data = {}
for treename in ["events/full/selected", "events/full/sideband"]:
    arrs = f_mc[treename].arrays(branches, library="np")
    for b in branches:
        data.setdefault(b, []).append(arrs[b])
data = {b: np.concatenate(v) for b, v in data.items()}

bkg_mask = np.isin(data["category"], BACKGROUND_CATEGORIES)
print(f"Combined selected+sideband background (category 4-7) raw entries: {bkg_mask.sum()} / {len(bkg_mask)} total")

In [ ]:
def binned_stats(q2, weight, edges):
    """Per-bin raw count, weighted sum, and weighted-histogram statistical
    uncertainty (sqrt(sum(w^2))), for a given set of bin edges."""
    rows = []
    for i in range(len(edges) - 1):
        lo, hi = edges[i], edges[i + 1]
        m = (q2 >= lo) & (q2 < hi)
        sumw = weight[m].sum()
        sumw2 = (weight[m] ** 2).sum()
        stat_unc = np.sqrt(sumw2)
        rel_unc_pct = 100 * stat_unc / sumw if sumw > 0 else float("nan")
        rows.append((lo, hi, m.sum(), sumw, stat_unc, rel_unc_pct))
    return rows

def print_stats(rows):
    print(f"{'bin':>18} | {'raw N':>7} | {'sum(w)':>10} | {'rel unc %':>9}")
    for lo, hi, raw_n, sumw, stat_unc, rel_unc_pct in rows:
        label = f"[{lo:.2f},{hi:.2f})" if hi < 9999 else f"[{lo:.2f}, inf)"
        print(f"{label:>18} | {raw_n:7d} | {sumw:10.2f} | {rel_unc_pct:9.2f}")

q2_bkg = data["true_generator_q2"][bkg_mask]
wt_bkg = data["ppfx_cv_weight"][bkg_mask]

print("=== current binning (8 x 0.1 GeV^2 + overflow) ===")
current_rows = binned_stats(q2_bkg, wt_bkg, CURRENT_EDGES)
print_stats(current_rows)
print()

print("=== 2x finer binning (16 x 0.05 GeV^2 + overflow) ===")
fine_edges = list(np.arange(0.0, 0.85, 0.05)) + [99999.0]
fine_rows = binned_stats(q2_bkg, wt_bkg, fine_edges)
print_stats(fine_rows)

worst_current = max(r[5] for r in current_rows)
worst_fine = max(r[5] for r in fine_rows)
print()
print(f"Worst-case relative MC stat. uncertainty -- current binning: {worst_current:.2f}%, 2x finer: {worst_fine:.2f}%")

**Interpretation:** even at 2x the resolution of the current binning, the worst-case bin still sits at 3.24% relative MC statistical uncertainty -- nowhere near a reasonable floor (e.g. 10%). So raw MC statistics is *not* the constraint that justifies the current 0.1 GeV$^2$ bin width; the current choice has a large safety margin, and could go finer from a pure-counting standpoint if there were a physical reason to. That physical reason (or the lack of one) is checked next.

## Step 2/3 continued: is `true_W` actually usable for a 2D binning like Howard's?

Howard bins his BackgroundFit parameters in (true $W$, true $Q^2$). Before even considering that, check whether `true_W` is populated for our own background events.

In [ ]:
w_bkg = data["true_W"][bkg_mask]
n_nan = np.isnan(w_bkg).sum()
print(f"true_W is NaN for {n_nan} / {bkg_mask.sum()} background events ({100*n_nan/bkg_mask.sum():.1f}%)")
if n_nan < bkg_mask.sum():
    print(f"true_W (non-NaN) range: {np.nanmin(w_bkg):.4f} - {np.nanmax(w_bkg):.4f}")

**Interpretation:** `true_W` is undefined for roughly 61-67% of our own background population (checked separately in `selected` and `sideband` in chat; the combined number is printed above). A 2D (W, Q$^2$) binning the way Howard does it isn't something we can drop in as-is with the branches we currently write out -- either most background events would fall into an undefined/overflow W bin, or `true_W` would need to be derived/imputed for the missing cases first. Sticking to 1D `true_generator_q2` binning is the supported choice given what's actually in the ntuple today, not an arbitrary simplification.

## Step 4: does the Q$^2$ binning have real separating power in the fit's actual observable?

Raw truth-level statistics being generous doesn't by itself mean the fit can actually *distinguish* the 9 bins from each other -- that depends on whether different true Q$^2$ bins populate different regions of the reco-level observable the fit actually bins in, which is `reco_dpT_lp` (see `fitSamples_reco_dpT_containment.yaml`), **not** `reco_Q2` (a different branch, reconstructed under an assumption that doesn't hold for non-signal topologies -- checked and discarded as the wrong comparison in chat: it showed `reco_Q2` values up to 598 with no visible correlation to `true_generator_q2` for background events, which is a red herring, not a resolution measurement, since `reco_Q2` plays no role in the actual fit).

Check the sideband+contained selection specifically (`cut_type==1 && reco_leading_muon_containment==1`), since that's the primary control region constraining these parameters.

In [ ]:
sel = bkg_mask & (data["cut_type"] == 1) & (data["reco_leading_muon_containment"] == 1)
print(f"n background events passing sideband+contained selection: {sel.sum()}")
print()

print(f"{'true q2 bin':>16} | {'N':>5} | {'mean reco_dpT_lp':>16} | {'std reco_dpT_lp':>15}")
means = []
for i in range(len(CURRENT_EDGES) - 1):
    lo, hi = CURRENT_EDGES[i], CURRENT_EDGES[i + 1]
    m = sel & (data["true_generator_q2"] >= lo) & (data["true_generator_q2"] < hi)
    if m.sum() < 2:
        continue
    d = data["reco_dpT_lp"][m]
    w = data["ppfx_cv_weight"][m]
    mean_d = np.average(d, weights=w)
    std_d = np.sqrt(np.average((d - mean_d) ** 2, weights=w))
    means.append(mean_d)
    label = f"[{lo:.1f},{hi:.1f})" if hi < 9999 else f"[{lo:.1f},inf)"
    print(f"{label:>16} | {m.sum():5d} | {mean_d:16.2f} | {std_d:15.2f}")

print()
print(f"Mean reco_dpT_lp rises from {means[0]:.1f} (lowest Q2 bin) to {means[-1]:.1f} (overflow) -- a factor of {means[-1]/means[0]:.2f}x.")

**Interpretation:** the mean of `reco_dpT_lp` rises monotonically and substantially (a ~1.4x shift end to end) as a function of the true Q$^2$ bin, even though each individual bin's own spread is broad and overlapping. That's a genuine, physically sensible trend (higher-$Q^2$ backgrounds tend to produce more energetic/wider-angle hadronic activity, pushing reconstructed $\delta p_T$ higher) -- not a flat, uncorrelated smear. So the fit has real (soft, aggregate) reco-level handle to tell these bins apart; the binning isn't just nominally fine relative to unlimited raw statistics, it's resolving something physically real, though the overlap also means finer-than-current binning would trade on progressively smaller between-bin mean shifts -- a point of diminishing returns that isn't pinned down exactly here.

## Step 5: prior width from an independent generator comparison (GENIE vs. NuWro)

The current flat 40% is Howard's *first, simplest* choice, not something derived for our analysis. His *most rigorous* option ("bin-by-bin uncertainties") came from a GENIE/NuWro ratio -- and we have local GENIE and NuWro NUISANCE flat-tree productions (`data/Generators/{GENIE,NuWro}/fhc_Nu14/output_*.nuisflat.root`, 10 files of 1M events each) that let us redo the same kind of comparison.

**Important caveat, stated plainly:** these are bare generator-truth samples with no ICARUS detector simulation or selection applied, so there's no direct branch equivalent to our detector-level categories 4-7 (which depend on what actually gets reconstructed/misidentified). The closest generator-truth proxy for "the physical process class this background represents" is *inclusive CC that is not CC0$\pi$* (`flagCCINC & ~flagCC0pi`, using the flat trees' own topology flags). This approximates, but does not exactly reproduce, categories 4-7. All 10 files per generator (10M events each) are used below -- an earlier pass with only 6/10 NuWro files (4 were cloud-only at the time) gave per-bin widths within 0.003 of the full-statistics numbers here, so this was already statistics-saturated; the full set just removes that caveat.

In [ ]:
GENERATORS_DIR = f'{ICARUS_ROOT}/ICARUS_CC0pi_GUNDAM/data/Generators'

def load_ccother(genname, indices):
    q2_list, ccother_list, ccinc_list = [], [], []
    for i in indices:
        path = f"{GENERATORS_DIR}/{genname}/fhc_Nu14/output_{genname}_{i}.nuisflat.root"
        t = open_with_retry(path)["FlatTree_VARS"]
        arrs = t.arrays(["Q2_true", "flagCCINC", "flagCC0pi"], library="np")
        ccinc = arrs["flagCCINC"].astype(bool)
        cc0pi = arrs["flagCC0pi"].astype(bool)
        q2_list.append(arrs["Q2_true"] / 1e6)  # Q2_true is in MeV^2
        ccother_list.append(ccinc & ~cc0pi)
        ccinc_list.append(ccinc)
    return np.concatenate(q2_list), np.concatenate(ccother_list), np.concatenate(ccinc_list)

q2_g, ccother_g, ccinc_g = load_ccother("GENIE", range(10))
q2_n, ccother_n, ccinc_n = load_ccother("NuWro", range(10))

print(f"GENIE: {len(q2_g)} events, CC-other/CC-inclusive = {ccother_g.sum()/ccinc_g.sum():.4f}")
print(f"NuWro: {len(q2_n)} events, CC-other/CC-inclusive = {ccother_n.sum()/ccinc_n.sum():.4f}")

In [ ]:
def ccother_fraction_per_bin(q2, is_ccother, is_ccinc, edges):
    """Fraction of ALL CC-inclusive interactions that are CC-other AND land in
    bin i -- normalizing by the total CC-inclusive count (not just CC-other)
    cancels any difference in how many raw events happen to be in each
    generator's file batch, while preserving each generator's own physical
    prediction for the CC-other/Q2 shape."""
    q2_ccinc = q2[is_ccinc]
    other_flag = is_ccother[is_ccinc]
    n_ccinc_total = is_ccinc.sum()
    fracs, raw_ns = [], []
    for i in range(len(edges) - 1):
        lo, hi = edges[i], edges[i + 1]
        m = other_flag & (q2_ccinc >= lo) & (q2_ccinc < hi)
        raw_ns.append(m.sum())
        fracs.append(m.sum() / n_ccinc_total)
    return np.array(fracs), np.array(raw_ns)

f_g, n_g = ccother_fraction_per_bin(q2_g, ccother_g, ccinc_g, CURRENT_EDGES)
f_n, n_n = ccother_fraction_per_bin(q2_n, ccother_n, ccinc_n, CURRENT_EDGES)
R = f_n / f_g
width_from_generators = np.abs(R - 1)

print(f"{'bin':>16} | {'N_GENIE':>8} | {'N_NuWro':>8} | {'f_GENIE':>9} | {'f_NuWro':>9} | {'R=fN/fG':>8} | {'|R-1|':>7}")
for i in range(len(CURRENT_EDGES) - 1):
    lo, hi = CURRENT_EDGES[i], CURRENT_EDGES[i + 1]
    label = f"[{lo:.1f},{hi:.1f})" if hi < 9999 else f"[{lo:.1f},inf)"
    print(f"{label:>16} | {n_g[i]:8d} | {n_n[i]:8d} | {f_g[i]:9.5f} | {f_n[i]:9.5f} | {R[i]:8.3f} | {width_from_generators[i]:7.3f}")

print()
print("Candidate bin-by-bin prior width (|R-1|):", np.round(width_from_generators, 3).tolist())
print("Current flat width for comparison:        ", [0.4] * len(width_from_generators))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bin_labels = [f"[{CURRENT_EDGES[i]:.1f},{CURRENT_EDGES[i+1]:.1f})" if CURRENT_EDGES[i+1] < 9999
              else f"[{CURRENT_EDGES[i]:.1f},inf)" for i in range(len(CURRENT_EDGES) - 1)]
x = np.arange(len(bin_labels))

ax.bar(x, width_from_generators, width=0.6, color='steelblue', alpha=0.8, label='GENIE-vs-NuWro |R-1|')
ax.axhline(0.4, color='salmon', linestyle='--', linewidth=1.5, label='current flat 40% prior')

ax.set_xticks(x)
ax.set_xticklabels(bin_labels, rotation=45, ha='right')
ax.set_xlabel(r'True $Q^2$ bin $\mathbf{[GeV^2]}$', fontsize=12, weight='bold')
ax.set_ylabel('Candidate fractional prior width', fontsize=12, weight='bold')
ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3)
fig.tight_layout()

import os
os.makedirs('../Plots/dpT/BackgroundTemplateDerivation', exist_ok=True)
fig.savefig('../Plots/dpT/BackgroundTemplateDerivation/genie_vs_nuwro_bin_width.pdf', bbox_inches='tight')

## Summary and recommendation

- **Binning:** keep the current 9 bins (8 x 0.1 GeV$^2$ + overflow). It's well inside the statistics floor (worst case 2.25% relative MC stat. uncertainty; even 2x finer only reaches 3.24%), a 2D (W,Q$^2$) binning isn't currently supportable given `true_W`'s ~61-67% NaN rate in our own background sample, and the binning shows genuine (not purely nominal) separating power in the fit's actual `reco_dpT_lp` observable.
- **Prior width:** the flat 40% is not well-motivated bin-by-bin. The GENIE-vs-NuWro comparison gives a strongly non-flat candidate width -- largest at the lowest Q$^2$ bin (42%, *exactly* the bin our Q2 fake-data closure test pulled hardest, to -0.70$\sigma$, under the current flat prior), smallest in the middle bins (as low as 1.6%), rising again toward the overflow bin (~20%). This is a materially different (and better-justified) shape than a single borrowed number, and it's directly checkable: whatever the low-Q$^2$ excess absorption in the existing closure test reflects, this result says GENIE and NuWro genuinely disagree most there, not that the prior happened to be loose there.
- **What this is *not*:** a replacement for Howard's approach, or a claim that this exact vector is final. The GENIE/NuWro comparison is a generator-truth-level proxy (no detector simulation, approximate category match), and the bin-separating-power check in Step 4 doesn't pin down an optimal bin width, just confirms the current one isn't degenerate. Before writing a new covariance from `width_from_generators`, the natural next step is exactly what Step 4 of the chat discussion outlined: rerun the fake-data closure tests with this bin-by-bin covariance in place of the flat 0.16 diagonal, and check whether the GENIE dial pulls (and the signal template) behave differently than they did under the flat 40% prior.